# Wasserstein + tube на вращающихся лучах: весь student training виден

Четыре режима из одного frozen unconditional FM-teacher: **M0 = MSE**, **M1 = MSE + tube**,
**E0 = MSE + Sinkhorn endpoint**, **E1 = MSE + Sinkhorn endpoint + tube**. Учитель делает два
Euler-микрошага, ученик — один: **32 → 16 NFE**. Время `t=1` означает шум, `t=0` — данные.

Ниже открыты teacher target, пересчёт target после tube, признаки, loss, optimizer,
`backward`, clipping и EMA. Sinkhorn — отдельная проверенная математическая функция;
в неё поступает **целая точка/изображение**, а не отдельные координаты или пиксели.
Прогон по умолчанию обучает четыре новых student-модели, один paired seed, по 200 updates.
Это интерактивный CPU-пилот и пример изменения метода; подтверждающая MNIST-серия и её
главный feature-MMD-контраст находятся в CLI-протоколе. Одна toy-серия не устанавливает
победителя для MNIST. Мы не подбираем результат по красивой картинке.

Веса учителя скачиваются **отдельными файлами из репозитория**. В ноутбуке нет wheel,
base64-весов или скрытого training runner. Новые checkpoints и GIF сохраняются рядом с запуском.

In [ ]:
import os
import subprocess
import sys

GIT_BRANCH = "research/wasserstein-tube-replication"
GIT_SOURCE = (
    "diffusion-lab[replication,interactive] @ "
    "git+https://github.com/Vovanm88/ddpm_vibe_research.git@" + GIT_BRANCH
)
if os.environ.get("DIFFUSION_LAB_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", GIT_SOURCE])


In [ ]:
import copy
import hashlib
import math
import tempfile
import time
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, clear_output, display

from diffusion_lab.visualization import plot_rainbow_trajectories, save_figure, save_trajectory_gif
from recheck.data import make_toy_dataset
from recheck.evaluate import fit_feature_reference, fixed_projection_bank, rays_quality
from recheck.fm import learning_rate, load_model, sample_euler
from recheck.losses import production_sinkhorn, sliced_wasserstein_loss
from recheck.runtime import RNGStreams, atomic_json, model_hash, save_checkpoint, tensor_hash


In [ ]:
DEVICE = "cpu"
THREADS = 4
SEED = 101
DATA_SEED = 41
N_CLASSES = 4
N_TRAIN = 8192
N_VALIDATION = 4096
UPDATES = 200
BATCH_SIZE = 128                     # also OT support size; no hidden accumulation
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP = 20
EMA_DECAY = 0.99
GRAD_CLIP = 1.0
TEACHER_STEPS = 32
STUDENT_STEPS = 16
TUBE_RHO = 0.10
DISTRIBUTION_FAMILY = "sinkhorn"      # "sinkhorn" or "swd"
REPRESENTATION = "endpoint"          # "endpoint", "displacement", "joint"
LAMBDA_W = 1.0
GAMMA = 1.0
EPSILON_ALPHA = 0.05
CALIBRATION_SAMPLES = 256
SWD_PROJECTIONS = 128
CHECKPOINT_EVERY = 50
RESUME = False                      # True restores exactly this run/config
EVAL_SAMPLES = 1024
EVAL_SEED = 730002                   # separate fixed validation noise, independent of NFE
GIF_PARTICLES = 16
MAKE_GIFS = True
SHOW_WIDGETS = True
OUTPUT_DIR = Path("outputs/wasserstein_tube_rays")


In [ ]:
# Find the checkout, or download its source/teacher separately from this notebook.
REPOSITORY_URL = "https://github.com/Vovanm88/ddpm_vibe_research.git"
repo_dir = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "src" / "recheck").is_dir()), None)
if repo_dir is None:
    repo_dir = Path(tempfile.mkdtemp(prefix="wasserstein-tube-source-")) / "repository"
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH,
                           REPOSITORY_URL, str(repo_dir)])

teacher_candidates = [
    repo_dir / "replication/demo/rays_cpu_pilot_v1/teacher/ema_model",
    repo_dir / "replication/artifacts/rays_cpu_pilot_v1/teacher/ema_model",
]
teacher_path = next((p for p in teacher_candidates if (p / "config.json").is_file()), None)
if teacher_path is None:
    raise FileNotFoundError("Run the documented rays teacher/pilot first, or use the published branch with demo weights")
torch.set_num_threads(THREADS)
device = torch.device(DEVICE)
if device.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable: choose DEVICE='cpu' for this notebook")
teacher = load_model(teacher_path, device).eval().requires_grad_(False)
teacher_hash = model_hash(teacher)
OUTPUT_DIR = Path(OUTPUT_DIR).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train = make_toy_dataset("rotating_rays", N_TRAIN, DATA_SEED, split="train", n_classes=N_CLASSES)
validation = make_toy_dataset("rotating_rays", N_VALIDATION, DATA_SEED,
                              split="validation", n_classes=N_CLASSES)
assert TEACHER_STEPS % STUDENT_STEPS == 0 and STUDENT_STEPS > 0
MICROSTEPS = TEACHER_STEPS // STUDENT_STEPS
H = 1.0 / STUDENT_STEPS
if DISTRIBUTION_FAMILY not in {"sinkhorn", "swd"}:
    raise ValueError("choose sinkhorn or swd")
if REPRESENTATION not in {"endpoint", "displacement", "joint"}:
    raise ValueError("unknown representation")
print("Teacher:", teacher_path, "parameters:", sum(p.numel() for p in teacher.parameters()))
print("Frozen teacher hash:", teacher_hash, "unconditional; labels never enter a model")
print("New student checkpoints and plots:", OUTPUT_DIR)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(validation[:, 0], validation[:, 1], s=2, alpha=.5)
ax.set(title="Fixed real validation · unconditional rotating rays", aspect="equal")
plt.show()


## Один переход: две скорости учителя → одна скорость ученика

`a_used` — один общий вход для обеих моделей. В tube-режимах `a_used = a + rho*H*eta`,
где `eta ~ N(0,I)` **на каждую координату**, без нормирования длины. Учитель каждый раз
заново интегрируется из `a_used`. Конец локального перехода не обязательно является
концом всей генерации. Ученик получает только `(x,t)`: длина `H` находится снаружи модели.

In [ ]:
@torch.no_grad()
def teacher_microsteps(model, anchor, t, width, microsteps=2):
    state = anchor.detach()
    h = width / microsteps
    for k in range(microsteps):
        velocity = model(state, float(t) - k * h).sample
        state = state - h * velocity                 # t decreases toward the data
    return state.detach()


def transition_features(anchor, endpoint, representation="endpoint", gamma=1.0):
    dimension = endpoint[0].numel()
    a = anchor.flatten(1)
    y = endpoint.flatten(1)
    if representation == "endpoint":
        return y / math.sqrt(dimension)
    displacement = y - a
    if representation == "displacement":
        return displacement / math.sqrt(dimension)
    if representation == "joint":
        return torch.cat((math.sqrt(gamma) * a, displacement), dim=1) / math.sqrt(dimension)
    raise ValueError("unknown representation")


def distribution_penalty(phi_student, phi_teacher, family, epsilon, streams):
    if family == "sinkhorn":
        # Canonical squared-cost, KL regularization, both student self-term gradients.
        # Raises if either marginal/solver fails; no clamp or nan_to_num shortcut.
        return production_sinkhorn(phi_student.float(), phi_teacher.float(), epsilon)
    if family == "swd":
        # New random directions use only their own stream, preserving paired data/t/noise.
        return sliced_wasserstein_loss(phi_student.float(), phi_teacher.float(),
                                      generator=streams["swd"], num_directions=SWD_PROJECTIONS)
    raise ValueError("unknown distribution family")


## Калибровка до обучения: только frozen teacher и чистый train

Для каждого из 16 интервалов и каждого representation вычисляем median off-diagonal
teacher-to-teacher squared cost, затем `epsilon = .05 * max(median, 1e-8)`.
Эта таблица одинакова для tube/no-tube, не зависит от student и не обновляется во время training.
Это train-only калибровка, а не подбор на validation или test.

In [ ]:
calibration_streams = RNGStreams(SEED + 10000)
calibration = {representation: [] for representation in ("endpoint", "displacement", "joint")}
calibration_trace = []
with torch.no_grad():
    for j in range(STUDENT_STEPS):
        t = 1.0 - j * H
        indices = torch.randint(len(train), (CALIBRATION_SAMPLES,), generator=calibration_streams["data"])
        x0 = train[indices].to(device)
        x1 = torch.randn(x0.shape, generator=calibration_streams["noise"]).to(device)
        anchor = (1 - t) * x0 + t * x1
        endpoint = teacher_microsteps(teacher, anchor, t, H, MICROSTEPS)
        calibration_trace.append(tensor_hash(anchor))
        for representation in calibration:
            phi_teacher = transition_features(anchor, endpoint, representation, GAMMA).detach().cpu().double()
            costs = torch.cdist(phi_teacher, phi_teacher).square()
            upper = torch.triu_indices(len(costs), len(costs), offset=1, device=costs.device)
            median_cost = float(torch.quantile(costs[upper[0], upper[1]], .5))
            if median_cost < 1e-8:
                print("WARNING: degenerate teacher cost", representation, j, median_cost)
            calibration[representation].append({"interval": j, "t": t, "H": H,
                "median_cost": median_cost, "epsilon": EPSILON_ALPHA * max(median_cost, 1e-8)})
atomic_json(OUTPUT_DIR / "calibration.json", {"representations": calibration,
    "clean_train_anchor_hashes": calibration_trace, "alpha": EPSILON_ALPHA,
    "teacher_sha256": teacher_hash, "support_size": CALIBRATION_SAMPLES})
print("Endpoint epsilon:", [round(row["epsilon"], 6) for row in calibration["endpoint"]])


## Открытый training loop: student init → target → loss → backward → optimizer → EMA

Все методы стартуют **точной копией одних и тех же teacher-весов**. Для каждого метода создаём
одинаковые независимые RNG streams: data, interval, noise, tube, SWD. Даже `rho=0` потребляет
тот же `eta`. Дополнительная SWD-случайность не меняет ни данные, ни времена, ни шумы.
`pairing_trace` проверяет идентичность последовательностей после прогона.

Главный checkpoint — финальная EMA после одинакового числа updates. Loss не делится на `H²`.
Линейные anchors ниже — интерполянты независимых пар data/noise; это не индивидуальные пути
истинного rollout учителя. Меняя источник anchors, нужно повторять всю сравниваемую четвёрку.

In [ ]:
# Alternate representation/family becomes explicitly named exploratory runs.
dist_label = "E" if (DISTRIBUTION_FAMILY, REPRESENTATION) == ("sinkhorn", "endpoint") else (
    DISTRIBUTION_FAMILY + "_" + REPRESENTATION + "_"
)
method_specs = [("M0", None, 0.0), ("M1", None, TUBE_RHO),
                (dist_label + "0", DISTRIBUTION_FAMILY, 0.0),
                (dist_label + "1", DISTRIBUTION_FAMILY, TUBE_RHO)]
trained, training_logs, pairing_traces, run_manifests = {}, {}, {}, {}

for method_name, family, rho in method_specs:
    student = copy.deepcopy(teacher).train().requires_grad_(True)
    ema = copy.deepcopy(teacher).eval().requires_grad_(False)
    assert model_hash(student) == teacher_hash == model_hash(ema)
    optimizer = torch.optim.AdamW(student.parameters(), lr=LEARNING_RATE,
                                  betas=(.9, .999), weight_decay=WEIGHT_DECAY)
    streams = RNGStreams(SEED)                     # RESET identically for every method
    method_dir = OUTPUT_DIR / method_name
    method_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = method_dir / "training.pt"
    contract = {"teacher_sha256": teacher_hash, "train_sha256": tensor_hash(train),
                "seed": SEED, "updates": UPDATES, "batch_size": BATCH_SIZE,
                "teacher_steps": TEACHER_STEPS, "student_steps": STUDENT_STEPS,
                "family": family, "representation": REPRESENTATION, "rho": rho,
                "lambda_w": LAMBDA_W, "gamma": GAMMA, "lr": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY, "warmup": WARMUP, "ema_decay": EMA_DECAY,
                "grad_clip": GRAD_CLIP, "swd_projections": SWD_PROJECTIONS,
                "calibration": calibration[REPRESENTATION], "anchor_source": "interpolant"}
    history, pairing_trace, completed = [], [], 0
    if checkpoint_path.exists():
        if not RESUME:
            raise FileExistsError(f"{checkpoint_path} exists: use RESUME=True or a new OUTPUT_DIR")
        saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        if saved["contract"] != contract:
            raise ValueError("Resume contract differs: use a new output directory")
        student.load_state_dict(saved["student"])
        ema.load_state_dict(saved["ema"])
        optimizer.load_state_dict(saved["optimizer"])
        for state in optimizer.state.values():
            for key, value in state.items():
                if isinstance(value, torch.Tensor):
                    state[key] = value.to(device)
        streams.load_state_dict(saved["streams"])
        torch.set_rng_state(saved["torch_rng"])
        if device.type == "cuda" and saved["cuda_rng"]:
            torch.cuda.set_rng_state_all(saved["cuda_rng"])
        history, pairing_trace = saved["history"], saved["pairing_trace"]
        completed = saved["updates_completed"]
    elif RESUME:
        print(method_name, "has no checkpoint: start its paired run from the teacher")

    def commit_checkpoint(status):
        save_checkpoint(checkpoint_path, {"contract": contract,
            "student": student.state_dict(), "ema": ema.state_dict(),
            "optimizer": optimizer.state_dict(), "streams": streams.state_dict(),
            "torch_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else [],
            "history": history, "pairing_trace": pairing_trace,
            "updates_completed": completed, "status": status})

    started = time.perf_counter()
    previous_seconds = history[-1]["training_seconds"] if history else 0.
    commit_checkpoint("running")
    try:
        for update in range(completed, UPDATES):
            j = int(torch.randint(STUDENT_STEPS, (), generator=streams["interval"]))
            t = 1.0 - j * H                        # ONE shared time for the whole OT support
            indices = torch.randint(len(train), (BATCH_SIZE,), generator=streams["data"])
            x0 = train[indices].to(device)
            x1 = torch.randn(x0.shape, generator=streams["noise"]).to(device)
            eta = torch.randn(x0.shape, generator=streams["tube"]).to(device)
            anchor = (1 - t) * x0 + t * x1
            a_used = anchor + rho * H * eta
            # RECOMPUTE the target FROM THIS perturbed input, never reuse a clean target.
            teacher_endpoint = teacher_microsteps(teacher, a_used, t, H, MICROSTEPS)
            student_endpoint = a_used - H * student(a_used, t).sample
            pair = (student_endpoint - teacher_endpoint.detach()).square().mean()
            dist = pair.new_zeros(())
            if family is not None and LAMBDA_W != 0:
                phi_student = transition_features(a_used, student_endpoint, REPRESENTATION, GAMMA)
                phi_teacher = transition_features(a_used, teacher_endpoint, REPRESENTATION, GAMMA).detach()
                dist = distribution_penalty(phi_student, phi_teacher, family,
                                             calibration[REPRESENTATION][j]["epsilon"], streams)
            loss = pair + LAMBDA_W * dist
            if not torch.isfinite(loss):
                raise FloatingPointError("non-finite loss: stop this series and inspect the cause")
            lr = learning_rate(update, UPDATES, WARMUP, LEARNING_RATE)
            for group in optimizer.param_groups:
                group["lr"] = lr
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP,
                                                       error_if_nonfinite=True)
            optimizer.step()
            with torch.no_grad():
                for ema_parameter, parameter in zip(ema.parameters(), student.parameters()):
                    ema_parameter.lerp_(parameter.detach(), 1 - EMA_DECAY)
                for ema_buffer, buffer in zip(ema.buffers(), student.buffers()):
                    ema_buffer.copy_(buffer)
            completed = update + 1
            pairing_trace.append(hashlib.sha256((str(j) + tensor_hash(indices) +
                                  tensor_hash(x1) + tensor_hash(eta)).encode()).hexdigest())
            history.append({"update": completed, "pair_loss": float(pair.detach()),
                "dist_loss": float(dist.detach()), "total_loss": float(loss.detach()),
                "grad_norm": float(grad_norm), "lr": lr, "interval": j, "t": t,
                "training_seconds": previous_seconds + time.perf_counter() - started})
            if completed == 1 or completed % CHECKPOINT_EVERY == 0:
                print(method_name, completed, "pair=", round(float(pair.detach()), 7),
                      "dist=", round(float(dist.detach()), 7))
                commit_checkpoint("running")
    except BaseException as error:
        # Keep the LAST FULLY COMMITTED weights/RNG; discard any partial update.
        saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        saved["status"] = "interrupted" if isinstance(error, KeyboardInterrupt) else "failed"
        save_checkpoint(checkpoint_path, saved)
        atomic_json(method_dir / "manifest.json", contract | {
            "method": method_name, "status": saved["status"],
            "updates_completed": saved["updates_completed"], "error": str(error),
            "recovery": "RESUME=True restores last fully committed update"})
        raise
    commit_checkpoint("completed")
    assert model_hash(teacher) == teacher_hash and all(p.grad is None for p in teacher.parameters())
    ema.save_pretrained(method_dir / "ema_model", safe_serialization=True)
    manifest = contract | {"method": method_name, "status": "completed",
        "updates_completed": completed, "ema_sha256": model_hash(ema),
        "rng_seeds": streams.seeds, "teacher_calls_per_update": MICROSTEPS,
        "teacher_calls_training": completed * MICROSTEPS,
        "micro_batch": BATCH_SIZE, "optimizer_effective_batch": BATCH_SIZE,
        "ot_support_size": BATCH_SIZE if family is not None else 0,
        "precision": "float32 network; float64 converged Sinkhorn",
        "training_seconds": history[-1]["training_seconds"]}
    atomic_json(method_dir / "manifest.json", manifest)
    atomic_json(method_dir / "history.json", history)
    trained[method_name] = ema
    training_logs[method_name] = history
    pairing_traces[method_name] = pairing_trace
    run_manifests[method_name] = manifest

assert all(trace == pairing_traces["M0"] for trace in pairing_traces.values())
print("Paired data/t/noise/tube traces match for all four methods; teacher unchanged.")


## Quality и fidelity — отдельные измерения

Все методы и обе frozen baseline получают **один и тот же initial noise**; NFE не участвует
в seed. Quality измеряем относительно независимых real validation: MMD² с train-only
RBF reference и фиксированный SWD. Отрицательную unbiased MMD² не обрезаем.
Fidelity — paired endpoint MSE относительно original teacher на 32 шагах.
Сходство с учителем не означает улучшение относительно данных.

In [ ]:
initial_noise = torch.randn(EVAL_SAMPLES, 2, generator=torch.Generator().manual_seed(EVAL_SEED))
reference = fit_feature_reference(train, seed=SEED + 20000, bandwidth_samples=2000,
                                  standardize=False)
projection_bank = fixed_projection_bank(2, 1024, seed=SEED + 20001)
teacher_label = f"Original N{TEACHER_STEPS}"
models_to_compare = {teacher_label: (teacher, TEACHER_STEPS),
                     f"Original N{STUDENT_STEPS}": (teacher, STUDENT_STEPS)}
models_to_compare.update({f"{name} N{STUDENT_STEPS}": (model, STUDENT_STEPS) for name, model in trained.items()})
generations, metrics = {}, []
for label, (model, steps) in models_to_compare.items():
    generated = sample_euler(model, initial_noise, steps=steps, record_history=True)
    generations[label] = generated
    quality = rays_quality(generated["samples"], validation, reference, projection_bank,
                           block_size=256)
    fidelity = float((generated["samples"] - generations[teacher_label]["samples"]).square().mean())
    row = {"model": label, "nfe": generated["nfe"], "initial_noise_sha256": generated["initial_noise_sha256"],
           "paired_mse_to_original_teacher": fidelity, **quality}
    metrics.append(row)
    print(label, "SWD=", round(row["sliced_wasserstein"], 6),
          "MMD2(real)=", round(row["mmd2_real"], 7), "paired MSE=", round(fidelity, 7))
assert len({row["initial_noise_sha256"] for row in metrics}) == 1
atomic_json(OUTPUT_DIR / "metrics.json", {"scope": "unconditional_rotating_rays_validation_one_paired_seed",
    "seed": SEED, "reference": reference.to_dict(), "rows": metrics})
np.savez_compressed(OUTPUT_DIR / "samples_and_histories.npz", initial_noise=initial_noise.numpy(),
                    **{label.replace(" ", "_"): result["history"].numpy()
                       for label, result in generations.items()})


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
plots = [("Real validation", validation)] + [(name, result["samples"]) for name, result in generations.items()]
limits = np.concatenate([points.numpy() for _, points in plots])
low, high = limits.min(0) - .08, limits.max(0) + .08
for ax, (label, points) in zip(axes.flat, plots):
    ax.scatter(points[:, 0], points[:, 1], s=2, alpha=.5)
    ax.set(title=label, xlim=(low[0], high[0]), ylim=(low[1], high[1]), aspect="equal")
for ax in axes.flat[len(plots):]:
    ax.axis("off")
save_figure(fig, OUTPUT_DIR / "generation_comparison")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for name, history in training_logs.items():
    axes[0].plot([r["update"] for r in history], [r["pair_loss"] for r in history], label=name)
    axes[1].plot([r["update"] for r in history], [r["dist_loss"] for r in history], label=name)
axes[0].set(title="Raw paired endpoint loss", xlabel="Optimizer updates", yscale="log")
axes[1].set(title="Distribution loss (zero for M0/M1)", xlabel="Optimizer updates")
axes[1].set_yscale("symlog", linthresh=1e-7)
for ax in axes[:2]:
    ax.legend()
axes[2].bar(range(len(metrics)), [r["sliced_wasserstein"] for r in metrics])
axes[2].set_xticks(range(len(metrics)), [r["model"] for r in metrics], rotation=45, ha="right")
axes[2].set(title="Quality relative to real validation", ylabel="Fixed-projection SWD ↓")
save_figure(fig, OUTPUT_DIR / "losses_and_quality")
plt.show()


## GIF и интерактивное проигрывание фактических траекторий

Красный — initial noise, фиолетовый — финальный endpoint. Каждая линия хранит историю **той же
частицы**; позиции не масштабируются для анимации. У оригинала N32 и учеников N16 разные
числа Euler-узлов, слайдер показывает реальные индексы и время выбранной истории.

In [ ]:
all_states = torch.cat([result["history"].flatten(0, 1) for result in generations.values()])
common_low, common_high = all_states.min(0).values - .1, all_states.max(0).values + .1
gif_paths = []
if MAKE_GIFS:
    for label, result in generations.items():
        paths = result["history"][:, :GIF_PARTICLES].permute(1, 0, 2)
        path = OUTPUT_DIR / (label.replace(" ", "_") + "_trajectories.gif")
        save_trajectory_gif(paths, path, fps=10, max_frames=65,
                            title=label + " · red noise → purple data",
                            start_label="Start · noise t=1", end_label="Finish · data t=0")
        gif_paths.append(str(path))
    display(Image(filename=gif_paths[0]))
    display(Image(filename=gif_paths[-1]))
print("GIF files:", gif_paths)

if SHOW_WIDGETS:
    selector = widgets.Dropdown(options=list(generations), description="Model")
    frame = widgets.IntSlider(value=0, min=0, max=TEACHER_STEPS, step=1,
                              description="Euler step", continuous_update=False)
    play = widgets.Play(value=0, min=0, max=TEACHER_STEPS, step=1, interval=120)
    link = widgets.jslink((play, "value"), (frame, "value"))
    canvas = widgets.Output()

    def render_frame(change=None):
        result = generations[selector.value]
        last = len(result["history"]) - 1
        frame.max = play.max = last
        index = min(frame.value, last)
        with canvas:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(6, 6))
            state = result["history"][index]
            ax.scatter(state[:, 0], state[:, 1], s=2, alpha=.3, color="gray")
            plot_rainbow_trajectories(result["history"][:, :GIF_PARTICLES].permute(1, 0, 2),
                progress=index / last, ax=ax,
                title=f"{selector.value} · actual Euler index {index}/{last} · t={float(result['times'][index]):.4f}",
                start_label="Start · noise t=1", end_label="Finish · data t=0")
            ax.set_xlim(float(common_low[0]), float(common_high[0]))
            ax.set_ylim(float(common_low[1]), float(common_high[1]))
            plt.show()

    selector.observe(render_frame, names="value")
    frame.observe(render_frame, names="value")
    display(widgets.VBox([widgets.HBox([selector, play, frame]), canvas]))
    render_frame()


## Перезагрузка и варианты эксперимента

Для SWD поменяйте `DISTRIBUTION_FAMILY="swd"`; для displacement/joint — `REPRESENTATION`.
Обучите всю четвёрку в **новом** `OUTPUT_DIR`: это exploratory-серия с собственными именами,
а не замена чисел основного endpoint-контраста. `LAMBDA_W=0` проверяет совпадение MSE-режимов
при общем RNG. Для 32→8 поменяйте `STUDENT_STEPS=8`: teacher target автоматически сделает
четыре микрошага, каждый student снова стартует из оригинального teacher.

Чтобы изменить модель, замените `student = copy.deepcopy(teacher)` на собственный `torch.nn.Module`
с `forward(x,t) -> .sample`, уберите проверку равенства init hash и явно назовите архитектурную
абляцию: она уже не является чистым сравнением четырёх loss-режимов исходного протокола.
Для равного числа параметров можно поменять общую архитектуру teacher/student и заново обучить
одного общего teacher через видимый FM-пример в предыдущих ноутбуках или CLI `train-teacher`.

Полный MNIST-комплекс запускается отдельными командами после rays gate и проверки анимаций,
с независимым CNN-оценщиком ≥99% real validation. Этот notebook не скачивает MNIST.

In [ ]:
first_name = next(iter(trained))
restored = load_model(OUTPUT_DIR / first_name / "ema_model", device).eval().requires_grad_(False)
restored_result = sample_euler(restored, initial_noise, steps=STUDENT_STEPS)
assert torch.equal(restored_result["samples"], generations[f"{first_name} N{STUDENT_STEPS}"]["samples"])
assert model_hash(teacher) == teacher_hash
print("Native Diffusers ModelMixin save/load matches; use sample_euler(..., steps=STUDENT_STEPS).")
print("Inspect metrics.json, per-method training.pt / ema_model / manifest.json, PNG/SVG and GIF under", OUTPUT_DIR)
